# 04_mueve_archivos

Mueve a backup los archivos del `id_proceso` actual después de que el pipeline terminó.

Responsabilidad:
- Leer `control_archivos`.
- Identificar los archivos asociados al `id_proceso` actual.
- Copiarlos desde `raw` hacia `backup/processed/{id_proceso}`.
- Validar tamaño de copia.
- Eliminar el archivo original en `raw`.

Este notebook usa solo el volume `backup` y las rutas `archivo_origen` registradas en control.

In [0]:
# COMMAND ----------
dbutils.widgets.removeAll()
dbutils.widgets.text("catalog_name", "smart_data_dev", "Catalog name")
dbutils.widgets.text("schema_name", "storage", "Schema name")
dbutils.widgets.text("id_proceso", "02ae3d8f-05ac-4af0-86e8-05762dc2db18", "Process id")

catalog_name = dbutils.widgets.get("catalog_name").strip()
schema_name = dbutils.widgets.get("schema_name").strip()
id_proceso = dbutils.widgets.get("id_proceso").strip()

if not all([catalog_name, schema_name, id_proceso]):
    raise ValueError("catalog_name, schema_name e id_proceso son obligatorios.")

tabla_control_sql = f"`{catalog_name}`.`{schema_name}`.`control_archivos`"
backup_base_path = f"/Volumes/{catalog_name}/{schema_name}/backup"
ruta_backup_proceso = f"{backup_base_path}/processed/{id_proceso}"
id_proceso_sql = id_proceso.replace("'", "''")

print(f"catalog_name: {catalog_name}")
print(f"schema_name: {schema_name}")
print(f"id_proceso: {id_proceso}")
print(f"ruta_backup_proceso: {ruta_backup_proceso}")

In [0]:
# COMMAND ----------

def obtiene_archivo(ruta):
    ruta = ruta.rstrip("/")
    carpeta, nombre = ruta.rsplit("/", 1)

    try:
        for archivo in dbutils.fs.ls(carpeta):
            if archivo.name.rstrip("/") == nombre:
                return archivo
        return None
    except Exception as error:
        mensaje = str(error).lower()
        if (
            "file not found" in mensaje
            or "path does not exist" in mensaje
            or "no such file" in mensaje
            or "filenotfoundexception" in mensaje
        ):
            return None
        raise

In [0]:
# COMMAND ----------

archivos = spark.sql(f"""
SELECT DISTINCT
    nombre_archivo,
    archivo_origen
FROM {tabla_control_sql}
WHERE id_proceso = '{id_proceso_sql}'
AND detalle IS NULL
""").collect()

if not archivos:
    mensaje = f"No se encontraron archivos para mover. id_proceso={id_proceso}. Mueve archivos termina sin cambios."
    print(mensaje)
    dbutils.notebook.exit(mensaje)

dbutils.fs.mkdirs(ruta_backup_proceso)

cantidad_movidos = 0
cantidad_ya_existian = 0

for archivo in archivos:
    archivo_origen = archivo["archivo_origen"]
    nombre_archivo = archivo["nombre_archivo"]
    archivo_backup = f"{ruta_backup_proceso}/{nombre_archivo}"

    info_origen = obtiene_archivo(archivo_origen)
    info_backup = obtiene_archivo(archivo_backup)

    if info_origen is None:
        if info_backup is not None:
            print(f"Archivo ya estaba en backup y no existe en raw: {nombre_archivo}")
            cantidad_ya_existian += 1
            continue
        raise FileNotFoundError(f"El archivo {nombre_archivo} no existe ni en raw ni en backup")

    if info_backup is None:
        copiado = dbutils.fs.cp(archivo_origen, archivo_backup)
        if not copiado:
            raise RuntimeError(f"No se pudo copiar {nombre_archivo} a backup")
        info_backup = obtiene_archivo(archivo_backup)

    if info_backup is None:
        raise RuntimeError(f"No se encontró {nombre_archivo} en backup después de copiarlo")

    if info_origen.size != info_backup.size:
        raise RuntimeError(
            f"Copia incorrecta para {nombre_archivo}. raw={info_origen.size}; backup={info_backup.size}"
        )

    eliminado = dbutils.fs.rm(archivo_origen)
    if not eliminado:
        raise RuntimeError(f"No se pudo eliminar {nombre_archivo} de raw")

    if obtiene_archivo(archivo_origen) is not None:
        raise RuntimeError(f"El archivo {nombre_archivo} continúa existiendo en raw")

    cantidad_movidos += 1

print(
    f"Archivos movidos a backup. id_proceso={id_proceso}; "
    f"movidos={cantidad_movidos}; ya_existian={cantidad_ya_existian}; total_control={len(archivos)}"
)